<a href="https://colab.research.google.com/github/gauravd12345/ml_proj/blob/main/skipgram/skipgram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
nltk.download('brown')


True
1
Tesla T4


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import numpy as np
from nltk.corpus import brown

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
vocab = np.unique(brown.words()).reshape(-1, 1)
words = brown.words()

window = 2
vocab_size = len(vocab)

### Using One-Hot Encoding

In [ ]:

# The below approach is highly inefficient because we use one-hot encoding.
# Since the vocab size is ~50,000 words, we have 50,000 sparse one-hot vectors
#     that are very inefficient to perform computations with and have no semantic meaning.
# Instead, we could better represent words using embeddings.


# from sklearn.preprocessing import OneHotEncoder
#
#
# enc = OneHotEncoder(handle_unknown='ignore') # one hot encoding for the entire vocabulary
# enc.fit(vocab)

# word = 'and'
# categories = enc.categories_
# test_enc = enc.transform([[word]]).toarray()
# test_word = enc.inverse_transform(test_enc)

# print("Categories: ", categories)
# print(f"One-Hot Vector for '{word}': ", test_enc)
# print("One-Hot Vector Shape: ", test_enc.shape)
# print("Mapping One-Hot Vector to Word: ", test_word)

# X, y = [], []
# for i in range(len(words)):
#     center, context = [words[i]], []

#     for j in range(window): # words before the center
#         if (i + j) - window >= 0:
#             context.append(words[(i + j) - window])

#     for j in range(window): # words after the center
#         if i + j + 1 < len(words):
#             context.append(words[i + j + 1])

#     X.append(center)
#     y.append(context)

In [ ]:
wtoi, itow = {}, {}
for i in range(len(vocab)): # mapping words to indicies
    wtoi[vocab[i].item()] = i
    itow[i] = vocab[i].item()

X, y = [], []
l, r = 0, (2 * window) + 1
for i in range(window, vocab_size - r):
    center = wtoi[words[i]]
    for j in range(i - window, i + window + 1):
        if j != i:
            context = wtoi[words[j]]
            X.append(center)
            y.append(context)
    l += 1
    r += 1

In [ ]:
from sklearn.model_selection import train_test_split

batch_size = 64
class SkipgramDataset(Dataset):
    def __init__(self, center, context):
        self.center = torch.tensor(center)
        self.context = torch.tensor(context)

    def __len__(self):
        return len(self.center)

    def __getitem__(self, idx):
        return self.center[idx], self.context[idx]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

training_set = SkipgramDataset(X_train, y_train)
testing_set = SkipgramDataset(X_test, y_test)
train_dataloader = DataLoader(training_set, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(testing_set, batch_size=batch_size, shuffle=True)

In [ ]:
embedding_dim = 100

class Skipgram(nn.Module):
    def __init__(self):
        super().__init__() # neural network inherits from nn.Module

        """ skipgram network layers """
        # self.model = nn.Sequential(
        #     nn.Linear(vocab_size, hidden),
        #     nn.Linear(vocab_size, vocab_size * 4)
        # )
        self.model = nn.Sequential(
            nn.Embedding(vocab_size, embedding_dim),
            nn.Linear(embedding_dim, vocab_size)
        )

    """ forward pass of data """
    def forward(self, x):
        logits = self.model(x)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Skipgram().to(device)
print(f"{model} \n Using {device} device")

Skipgram(
  (model): Sequential(
    (0): Embedding(56057, 100)
    (1): Linear(in_features=100, out_features=56057, bias=True)
  )
) 
 Using cuda device


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    total_loss = 0.0

    for center, context in train_dataloader:
        optimizer.zero_grad()

        center = center.to(device)
        context = context.to(device)

        out = model(center)
        loss = criterion(out, context)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f'Epoch [{epoch + 1}/{epochs}], Avg Loss: {avg_loss:.4f}')

Epoch [1/50], Avg Loss: 4.9008
Epoch [2/50], Avg Loss: 4.7822
Epoch [3/50], Avg Loss: 4.7428
Epoch [4/50], Avg Loss: 4.7248
Epoch [5/50], Avg Loss: 4.7136
Epoch [6/50], Avg Loss: 4.7054
Epoch [7/50], Avg Loss: 4.6999
Epoch [8/50], Avg Loss: 4.6947
Epoch [9/50], Avg Loss: 4.6912
Epoch [10/50], Avg Loss: 4.6871
Epoch [11/50], Avg Loss: 4.6845
Epoch [12/50], Avg Loss: 4.6815
Epoch [13/50], Avg Loss: 4.6795
Epoch [14/50], Avg Loss: 4.6763
Epoch [15/50], Avg Loss: 4.6746
Epoch [16/50], Avg Loss: 4.6724
Epoch [17/50], Avg Loss: 4.6713
Epoch [18/50], Avg Loss: 4.6697
Epoch [19/50], Avg Loss: 4.6672
Epoch [20/50], Avg Loss: 4.6664
Epoch [21/50], Avg Loss: 4.6655
Epoch [22/50], Avg Loss: 4.6634
Epoch [23/50], Avg Loss: 4.6621
Epoch [24/50], Avg Loss: 4.6615
Epoch [25/50], Avg Loss: 4.6600
Epoch [26/50], Avg Loss: 4.6593
Epoch [27/50], Avg Loss: 4.6583
Epoch [28/50], Avg Loss: 4.6573
Epoch [29/50], Avg Loss: 4.6564
Epoch [30/50], Avg Loss: 4.6553
Epoch [31/50], Avg Loss: 4.6544
Epoch [32/50], Av

In [ ]:
word = itow[int(np.random.uniform(vocab_size))]
data = torch.tensor([wtoi[word]], dtype=torch.long).to(device)

with torch.no_grad():
    logits = model(data)
    topk = torch.topk(logits, 10, dim=1).indices[0].tolist()

print("Center word:", word)
print("Top context predictions:")
for idx in topk:
    print(itow[idx])

Center word: distrusted
Top context predictions:
following
Juvenile
decades
gave
The
rightfield
back
saying
Relations
motel
